# Isolated Reformulate Agent Evaluation

This notebook evaluates `src/agents/reformulate.py` in isolation on the first 20 queries of the MS MARCO dev2 topic file.

For each query:
1. Retrieve top-50 documents with the original query (the **initial retrieval**).
2. Ask the `ReformulationAgent` to rewrite the query.
3. Retrieve top-50 documents with the rewritten query.
4. Compute NDCG@10 for both rankings using the dev2 qrels.

Queries are then grouped by whether reformulation **helped**, left **unchanged**, or **harmed** NDCG@10.  Statistics are reported for the *initially retrieved* documents of each group.

**Abbreviations used throughout**
- `NDCG`: Normalized Discounted Cumulative Gain, a ranking quality metric (1 = perfect, 0 = none).
- `TF`: Term Frequency – how often a query term occurs in the retrieved corpus.
- `DF`: Document Frequency – in how many of the retrieved documents a query term appears.
- `df_cv`: Coefficient of variation of DF values (`std / mean`). Higher values mean the query terms are more unevenly distributed across the initial documents (more dispersion).
- `niche_term_ratio`: Fraction of query terms that appear in at most 1 initial document.
- `RR`: Reciprocal Rank, `1 / rank_of_first_relevant_document`.
- `semantic_similarity`: Average cosine similarity between the query embedding and the embeddings of the initially retrieved documents. Range [-1, 1]; higher values indicate stronger semantic alignment (a smaller semantic gap).

In [1]:
# -----------------------------------------------------------------------------
# CELL 1: Configuration
# -----------------------------------------------------------------------------
from pathlib import Path

# Project root is one level above this notebook.
ROOT = Path.cwd().parent

TOPICS_PATH = ROOT / "notebooks" / "queries" / "topics.ms-marco-dev2.tsv"
QRELS_PATH = ROOT / "notebooks" / "qrels" / "qrels.ms-marco-dev2.tsv"
OUTPUT_CSV = ROOT / "notebooks" / "reformulate_isolated_results.csv"

TOP_K = 50          # Number of documents retrieved per query.
NDCG_K = 10         # Cut-off used for NDCG and for grouping queries.
NUM_QUERIES = 20    # First N queries from the topics file.

# Threshold for deciding whether NDCG changed (guards against float noise).
DELTA_EPS = 1e-4

# A query term is considered "niche" in the initial corpus if it appears in at most
# NICHE_DF_THRESHOLD documents.
NICHE_DF_THRESHOLD = 1

## Imports and environment setup

In [2]:
# -----------------------------------------------------------------------------
# CELL 2: Imports
# -----------------------------------------------------------------------------
import re
import sys
import warnings
from collections import Counter, defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# Suppress noisy HTTPS certificate warnings from the OpenSearch retriever.
from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter("ignore", InsecureRequestWarning)

# Add project src/ to the Python path.
sys.path.insert(0, str(ROOT / "src"))

from sentence_transformers import SentenceTransformer
from src.agents.reformulate import ReformulationAgent
from src.utils.retriever import Retriever, create_retriever_callable

load_dotenv()

c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Load queries and qrels

In [3]:
# -----------------------------------------------------------------------------
# CELL 3: Load queries and qrels
# -----------------------------------------------------------------------------
def load_queries(path: Path, n: int) -> List[Tuple[str, str]]:
    queries = []
    with path.open("r", encoding="utf-8") as f:
        next(f)  # skip header: query_id\tquery_text
        for i, line in enumerate(f):
            if i >= n:
                break
            qid, qtext = line.strip().split("\t", 1)
            queries.append((qid, qtext))
    return queries


def load_qrels(path: Path) -> Dict[str, Dict[str, int]]:
    """Load qrels as {query_id: {doc_id: relevance}}."""
    qrels = defaultdict(dict)
    with path.open("r", encoding="utf-8") as f:
        next(f)  # skip header
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            qid, _, doc_id, rel = parts[0], parts[1], parts[2], parts[3]
            qrels[qid][doc_id] = int(rel)
    return qrels


queries = load_queries(TOPICS_PATH, NUM_QUERIES)
qrels = load_qrels(QRELS_PATH)
print(f"Loaded {len(queries)} queries and {sum(len(v) for v in qrels.values())} qrel entries.")

Loaded 20 queries and 5177 qrel entries.


## Helper functions

In [5]:
# -----------------------------------------------------------------------------
# CELL 4: NDCG, corpus-statistics, and semantic-gap helpers
# -----------------------------------------------------------------------------
def normalize_doc_id(doc_id: str) -> str:
    """Strip passage-level suffix (#...) to match qrel document identifiers."""
    return doc_id.split("#")[0] if "#" in doc_id else doc_id


def compute_dcg(relevances: List[int], k: int) -> float:
    """DCG@k with the standard (2^rel - 1) / log2(rank + 1) formulation."""
    dcg = 0.0
    for i, rel in enumerate(relevances[:k]):
        if rel > 0:
            # i=0 corresponds to rank 1, hence log2(i + 2).
            dcg += (2 ** rel - 1) / np.log2(i + 2)
    return dcg


def deduplicate_relevance(doc_ids: List[str], graded: List[int]) -> List[int]:
    """
    Deduplicate passage-level retrieval results to document-level relevance.
    The first occurrence of a normalized document keeps its relevance; later
    duplicates are treated as non-relevant (0).
    """
    seen = set()
    dedup = []
    for did, rel in zip(doc_ids, graded):
        norm = normalize_doc_id(did)
        if norm not in seen:
            dedup.append(rel)
            seen.add(norm)
        else:
            dedup.append(0)
    return dedup


def compute_ndcg(doc_ids: List[str], qid: str, k: int) -> float:
    """Compute NDCG@k for a ranked list of document IDs."""
    graded = [qrels[qid].get(normalize_doc_id(d), 0) for d in doc_ids]
    graded = deduplicate_relevance(doc_ids, graded)
    dcg = compute_dcg(graded, k)

    ideal = sorted(qrels[qid].values(), reverse=True)
    ideal = ideal + [0] * (k - len(ideal))
    idcg = compute_dcg(ideal, k)

    return dcg / idcg if idcg > 0 else 0.0


def tokenize(text: str) -> List[str]:
    """Simple regex tokenizer: lowercase, keep alphanumeric tokens."""
    return re.findall(r"\b[a-z0-9]+\b", text.lower())


def compute_initial_corpus_stats(query_text: str, doc_texts: List[str]) -> Dict[str, float]:
    """
    Compute corpus statistics for the initially retrieved (baseline) documents.
    
    Returns:
      - query_length_terms: number of unique query terms
      - num_docs: number of retrieved documents
      - coverage: fraction of query terms that appear anywhere in the corpus
      - mean_tf / median_tf / min_tf / max_tf: summary of query-term TFs
      - mean_df / median_df / min_df / max_df: summary of query-term DFs
      - df_cv: coefficient of variation of DF values (dispersion)
      - niche_term_ratio: fraction of query terms with DF <= 1
    """
    q_terms = set(tokenize(query_text))
    if not q_terms or not doc_texts:
        return {
            "query_length_terms": len(q_terms),
            "num_docs": len(doc_texts),
            "coverage": np.nan,
            "mean_tf": np.nan,
            "median_tf": np.nan,
            "min_tf": np.nan,
            "max_tf": np.nan,
            "mean_df": np.nan,
            "median_df": np.nan,
            "min_df": np.nan,
            "max_df": np.nan,
            "df_cv": np.nan,
            "niche_term_ratio": np.nan,
        }

    doc_tokens = [tokenize(t) for t in doc_texts]
    corpus_tokens = [tok for tokens in doc_tokens for tok in tokens]
    corpus_term_set = set(corpus_tokens)
    tf_counter = Counter(corpus_tokens)

    dfs = []
    tfs = []
    for term in q_terms:
        df = sum(1 for tokens in doc_tokens if term in tokens)
        tf = tf_counter.get(term, 0)
        dfs.append(df)
        tfs.append(tf)

    dfs = np.array(dfs, dtype=float)
    tfs = np.array(tfs, dtype=float)

    coverage = sum(1 for term in q_terms if term in corpus_term_set) / len(q_terms)

    df_mean = np.mean(dfs)
    df_cv = np.std(dfs) / df_mean if df_mean > 0 else 0.0
    niche_ratio = np.sum(dfs <= NICHE_DF_THRESHOLD) / len(q_terms)

    return {
        "query_length_terms": len(q_terms),
        "num_docs": len(doc_texts),
        "coverage": coverage,
        "mean_tf": np.mean(tfs),
        "median_tf": np.median(tfs),
        "min_tf": np.min(tfs),
        "max_tf": np.max(tfs),
        "mean_df": df_mean,
        "median_df": np.median(dfs),
        "min_df": np.min(dfs),
        "max_df": np.max(dfs),
        "df_cv": df_cv,
        "niche_term_ratio": niche_ratio,
    }


def compute_semantic_similarity(query_text: str, doc_texts: List[str], embed_model) -> float:
    """
    Compute the average cosine similarity between the query embedding and the
    embeddings of the initially retrieved documents.

    This quantifies the semantic gap between the query and the BM25-retrieved
    corpus: higher values mean the retrieved documents are semantically closer
    to the query (smaller gap).
    """
    if not doc_texts:
        return np.nan
    query_embedding = embed_model.encode([query_text], show_progress_bar=False)
    doc_embeddings = embed_model.encode(doc_texts, show_progress_bar=False)
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    return float(np.mean(similarities))

## Initialize retriever and ReformulationAgent

In [6]:
# -----------------------------------------------------------------------------
# CELL 5: Initialize models and retriever
# -----------------------------------------------------------------------------
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
retriever_instance = Retriever()
retriever_func = create_retriever_callable(retriever_instance)
agent = ReformulationAgent(embed_model=encoder)
print("Retriever and ReformulationAgent ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4825.83it/s]


Retriever and ReformulationAgent ready.


## Run evaluation

In [7]:
# -----------------------------------------------------------------------------
# CELL 6: Evaluate each query
# -----------------------------------------------------------------------------
records = []

for idx, (qid, qtext) in enumerate(queries, start=1):
    print(f"[{idx}/{len(queries)}] Query {qid}: {qtext}")

    # Baseline retrieval (this is the "initially retrieved" corpus we analyse).
    base_doc_ids, base_scores, base_corpus = retriever_func(qtext, TOP_K)
    base_ndcg = compute_ndcg(base_doc_ids, qid, NDCG_K)

    # Reformulate and retrieve.
    query_features = {
        "query_text": qtext,
        "retriever": retriever_func,
        "top_k": TOP_K,
    }
    effects = agent.compute_effects(query_features)
    reform_text = effects.get("new_query_text", qtext)
    reform_doc_ids = effects.get("new_doc_ids", [])
    reform_ndcg = compute_ndcg(reform_doc_ids, qid, NDCG_K)

    delta_ndcg = reform_ndcg - base_ndcg

    # Relevance stats for the initial retrieval (deduplicated).
    base_rels = deduplicate_relevance(
        base_doc_ids,
        [qrels[qid].get(normalize_doc_id(d), 0) for d in base_doc_ids]
    )
    rel_top10 = sum(1 for r in base_rels[:NDCG_K] if r > 0)
    rel_top50 = sum(1 for r in base_rels if r > 0)
    first_rel_rank = next((i + 1 for i, r in enumerate(base_rels) if r > 0), None)
    rr = 1.0 / first_rel_rank if first_rel_rank else 0.0

    # Corpus-level statistics for the initial retrieval.
    base_doc_texts = [base_corpus[d] for d in base_doc_ids]
    corpus_stats = compute_initial_corpus_stats(qtext, base_doc_texts)

    # Semantic similarity between the query and the initially retrieved corpus.
    semantic_similarity = compute_semantic_similarity(qtext, base_doc_texts, encoder)

    records.append({
        "qid": qid,
        "query": qtext,
        "reformulated_query": reform_text,
        "base_ndcg": base_ndcg,
        "reform_ndcg": reform_ndcg,
        "delta_ndcg": delta_ndcg,
        "rel_top10": rel_top10,
        "rel_top50": rel_top50,
        "first_rel_rank": first_rel_rank if first_rel_rank else float("inf"),
        "rr": rr,
        "semantic_similarity": semantic_similarity,
        **corpus_stats,
    })
    print(f"  base NDCG@{NDCG_K}={base_ndcg:.4f} | reform NDCG@{NDCG_K}={reform_ndcg:.4f} | delta={delta_ndcg:+.4f}")

df = pd.DataFrame(records)
print(f"\nEvaluated {len(df)} queries.")

[1/20] Query 1048579: what is pcnt
  base NDCG@10=1.0000 | reform NDCG@10=0.0000 | delta=-1.0000
[2/20] Query 262156: how long is a college hockey game
  base NDCG@10=0.5000 | reform NDCG@10=0.0000 | delta=-0.5000
[3/20] Query 1048601: what is pastoral medicine
  base NDCG@10=0.4307 | reform NDCG@10=0.0000 | delta=-0.4307
[4/20] Query 1048673: what is ownership of a corporation called
  base NDCG@10=0.0000 | reform NDCG@10=0.0000 | delta=+0.0000
[5/20] Query 786531: what is prevail
  base NDCG@10=0.0000 | reform NDCG@10=0.0000 | delta=+0.0000
[6/20] Query 1048706: what is overhead rate in managerial accounting?
  base NDCG@10=0.0000 | reform NDCG@10=0.0000 | delta=+0.0000
[7/20] Query 786568: what is price of pressure treated lumber 2x6x8
  base NDCG@10=0.3155 | reform NDCG@10=0.3010 | delta=-0.0144
[8/20] Query 1048730: what is outlook data file
  base NDCG@10=0.0000 | reform NDCG@10=0.0000 | delta=+0.0000
[9/20] Query 262330: how long is a flight from chicago to australia
  base NDCG

## Group queries by NDCG change

In [8]:
# -----------------------------------------------------------------------------
# CELL 7: Split into helped / unchanged / harmed
# -----------------------------------------------------------------------------
helped = df[df["delta_ndcg"] > DELTA_EPS].copy()
unchanged = df[df["delta_ndcg"].abs() <= DELTA_EPS].copy()
harmed = df[df["delta_ndcg"] < -DELTA_EPS].copy()

print("Group counts:")
print(f"  Helped:    {len(helped)} / {len(df)}")
print(f"  Unchanged: {len(unchanged)} / {len(df)}")
print(f"  Harmed:    {len(harmed)} / {len(df)}")

Group counts:
  Helped:    1 / 20
  Unchanged: 11 / 20
  Harmed:    8 / 20


## Per-group statistics for initially retrieved documents

In [10]:
# -----------------------------------------------------------------------------
# CELL 8: Aggregate statistics per group
# -----------------------------------------------------------------------------
stats_cols = [
    "base_ndcg",
    "rel_top10",
    "rel_top50",
    "first_rel_rank",
    "rr",
    "semantic_similarity",
    "query_length_terms",
    "num_docs",
    "coverage",
    "mean_tf",
    "median_tf",
    "min_tf",
    "max_tf",
    "mean_df",
    "median_df",
    "min_df",
    "max_df",
    "df_cv",
    "niche_term_ratio",
]

def group_summary(group_df: pd.DataFrame, name: str) -> pd.DataFrame:
    if group_df.empty:
        print(f"\n{name}: no queries in this group.")
        return pd.DataFrame()

    rows = []
    for col in stats_cols:
        values = group_df[col].astype(float)
        rows.append({
            "metric": col,
            "mean": values.mean(),
            "std": values.std(ddof=0),
            "min": values.min(),
            "median": values.median(),
            "max": values.max(),
        })
    return pd.DataFrame(rows)


for name, group in [("Helped", helped), ("Unchanged", unchanged), ("Harmed", harmed)]:
    summary = group_summary(group, name)
    if not summary.empty:
        print(f"\n{name} ({len(group)} queries):")
        display(summary.style.format(
                {
                    "mean": "{:.4f}",
                    "std": "{:.4f}",
                    "min": "{:.4f}",
                    "median": "{:.4f}",
                    "max": "{:.4f}",
                }
            )
        )


Helped (1 queries):


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,metric,mean,std,min,median,max
0,base_ndcg,0.0000,0.0000,0.0000,0.0000,0.0000
1,rel_top10,0.0000,0.0000,0.0000,0.0000,0.0000
2,rel_top50,0.0000,0.0000,0.0000,0.0000,0.0000
3,first_rel_rank,inf,nan,inf,inf,inf
4,rr,0.0000,0.0000,0.0000,0.0000,0.0000
5,semantic_similarity,0.3779,0.0000,0.3779,0.3779,0.3779
6,query_length_terms,6.0000,0.0000,6.0000,6.0000,6.0000
7,num_docs,50.0000,0.0000,50.0000,50.0000,50.0000
8,coverage,1.0000,0.0000,1.0000,1.0000,1.0000
9,mean_tf,1124.3333,0.0000,1124.3333,1124.3333,1124.3333



Unchanged (11 queries):


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,metric,mean,std,min,median,max
0,base_ndcg,0.1818,0.3857,0.0000,0.0000,1.0000
1,rel_top10,0.1818,0.3857,0.0000,0.0000,1.0000
2,rel_top50,0.2727,0.4454,0.0000,0.0000,1.0000
3,first_rel_rank,inf,nan,1.0000,inf,inf
4,rr,0.1837,0.3848,0.0000,0.0000,1.0000
5,semantic_similarity,0.5172,0.0604,0.4329,0.5222,0.6114
6,query_length_terms,5.1818,2.0810,2.0000,5.0000,9.0000
7,num_docs,50.0000,0.0000,50.0000,50.0000,50.0000
8,coverage,1.0000,0.0000,1.0000,1.0000,1.0000
9,mean_tf,206.8952,51.5048,135.3333,189.5000,304.2000



Harmed (8 queries):


,metric,mean,std,min,median,max
0,base_ndcg,0.5185,0.2072,0.2891,0.5000,1.0000
1,rel_top10,1.0000,0.0000,1.0000,1.0000,1.0000
2,rel_top50,1.0000,0.0000,1.0000,1.0000,1.0000
3,first_rel_rank,4.1250,3.0182,1.0000,3.0000,10.0000
4,rr,0.4344,0.3376,0.1000,0.3333,1.0000
5,semantic_similarity,0.4685,0.1113,0.3117,0.4498,0.6451
6,query_length_terms,4.8750,1.9645,3.0000,4.0000,8.0000
7,num_docs,50.0000,0.0000,50.0000,50.0000,50.0000
8,coverage,0.9821,0.0472,0.8571,1.0000,1.0000
9,mean_tf,152.7723,46.1088,105.3333,145.4762,237.1429


## Per-query semantic similarity

In [16]:
# -----------------------------------------------------------------------------
# CELL 9: Display average semantic similarity per query
# -----------------------------------------------------------------------------
semantic_display = harmed[["qid", "query", "semantic_similarity"]].copy()
semantic_display = semantic_display.sort_values("semantic_similarity", ascending=False)
display(semantic_display.style.format({"semantic_similarity": "{:.4f}"}))

,qid,query,semantic_similarity
6,786568,what is price of pressure treated lumber 2x6x8,0.6451
1,262156,how long is a college hockey game,0.6261
11,1048848,what is oprah winfrey's net wo,0.5064
2,1048601,what is pastoral medicine,0.4643
17,525186,tsa wages and benefits,0.4353
12,524574,trending topic meaning,0.3916
15,524827,triptans minimum age,0.3679
0,1048579,what is pcnt,0.3117


## Per-query details and CSV export

In [ ]:
# -----------------------------------------------------------------------------
# CELL 10: Show per-query table and save results
# -----------------------------------------------------------------------------
display_cols = [
    "qid", "query", "reformulated_query", "base_ndcg", "reform_ndcg",
    "delta_ndcg", "rel_top10", "rel_top50", "first_rel_rank",
    "semantic_similarity", "coverage", "mean_df", "df_cv", "niche_term_ratio"
]
display(df[display_cols].style.format({"base_ndcg": "{:.4f}", "reform_ndcg": "{:.4f}", "delta_ndcg": "{:+.4f}", "semantic_similarity": "{:.4f}"}))

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved per-query CSV to: {OUTPUT_CSV}")